# Multimodal Cancer Classification Challenge 2026 — v22 (heterogeneous ensemble)

**Goal: top the LB (0.79+).** No new training. Just load all trained ckpts (v19 EffNet-B0, v21 ResNet-50, optionally future v23/v24+) and average their predictions cell-by-cell after AdaBN + 8-way D4 TTA.

## LB context (May 17, 2026)

| # | Team | LB | Gap |
|---|---|---|---|
| 1 | Group 1 | 0.7832 | +0.038 ahead |
| 2 | Group 2 | 0.7713 | +0.026 ahead |
| **3** | **Rafael** | **v19 0.7455** | — |

L4 lecture slide 58: *"Train multiple independent models. At test time average their results. Enjoy 2% extra performance."* If we average 2–3 architecturally-different models, expected lift is **+0.015 to +0.035** — exactly enough to top.

## Why v22 is the ensemble, not a single model

v19 (EffNet-B0 @ 128) and v21 (ResNet-50 @ 224) are the two strongest **single-model** runs in our history. v22 is the natural next step: **combine them at inference**. No new training needed for the basic v22 — the ckpts already exist.

If the v19+v21 ensemble doesn't top the LB on its own, we extend v22 by adding a 3rd architecturally-different model (e.g. DenseNet-201, trained as v23). Then v22-extended = ensemble(v19, v21, v23).

## Models we combine

Set the `MODELS` list in the config cell to whatever you have. The notebook silently skips any model whose checkpoint doesn't exist.

| Default model | Backbone | Input | Source ckpt | Expected (alone) |
|---|---|---|---|---|
| v19 | EfficientNet-B0 (1-ch conv_stem) | 128 native | `/kaggle/input/v19-ckpt/fulldata_best.pt` | 0.7455 ✅ |
| v21 | ResNet-50 (1-ch conv1) | 224 upscaled | `/kaggle/input/v21-ckpt/fulldata_best.pt` | TBD (training now) |
| v23 (later) | DenseNet-201 OR early-fusion ResNet-50 | 128 OR 224 | `/kaggle/input/v23-ckpt/fulldata_best.pt` | TBD |

Each model runs:
1. **AdaBN pass** on test set (recalibrates BN to test distribution)
2. **8-way D4 TTA** at its native training resolution
3. Produces per-cell `sigmoid(logit)` predictions in [0, 1]

## Three submissions written (one per combination strategy)

You get **4 daily Kaggle submissions** — submit all three to see which works best:

| File | Combination | When it wins |
|---|---|---|
| `submission_sigmoid_avg.csv` | simple arithmetic mean of sigmoid outputs | usually the default, well-calibrated models |
| `submission_rank_avg.csv` | convert each model's scores to ranks within test, average ranks, normalize to [0,1] | when models have different scales (different temperature/confidence) |
| `submission_geomean.csv` | geometric mean of sigmoid outputs | preserves bimodal extremes; good when one model is decisive and another is uncertain |

## Compute on Kaggle T4

| Stage | Time |
|---|---|
| JPEG cache | ~50 min |
| Test pixel stats (for stain norm) | ~30 s |
| Inference per model: AdaBN + 8-way D4 TTA | ~5–10 min depending on input size |
| 2-model ensemble (v19 + v21) | **~65 min total** |
| 3-model ensemble (+ v23) | **~75 min total** |
| 4-model ensemble | **~85 min total** |

## Setup checklist

Before running this notebook:

1. **Attach the dataset**: `rafaelproena/a3-adl` (we've used this for v19/v21)
2. **Upload v19 ckpt as a Kaggle dataset**: in Kaggle UI, create a new dataset, upload `fulldata_best.pt` from `results (1) v19/runs/`. Title: `v19-ckpt`. Attach it.
3. **Upload v21 ckpt** (after v21 finishes): same procedure. Title: `v21-ckpt`.
4. *(Optional)* Upload future v23 ckpt when ready: `v23-ckpt`.

The notebook auto-discovers checkpoints; if a model's ckpt isn't found, that model is silently skipped (so v22 can run on just v19+v21 with no edits).

## Quick alternative — local CSV averaging (no Kaggle needed)

If you don't want to re-run inference, you can just average the **already-existing** `submission.csv` files from each version. We have a standalone `vEnsemble_csv.py` script next to this notebook for exactly that:

```bash
python vEnsemble_csv.py \
    "results/results (1) v19/submission.csv" \
    "path/to/v21/submission.csv" \
    --weights 0.7455 <v21_LB> \
    --out-dir ./v22_outputs
```

Faster than re-running inference but slightly less flexible (cannot re-AdaBN, cannot change TTA scheme).

## Workflow

1. v19 LB 0.7455 already submitted ✅
2. Wait for v21 to finish, download submission.csv, submit its LB
3. **Run `vEnsemble_csv.py` locally** on v19 + v21 submissions → three combination files
4. **Submit the best of {sigmoid_avg, rank_avg, geomean}** (this is v22)
5. If still not topping: train v23 (DenseNet-201 or early-fusion ResNet-50), then re-run `vEnsemble_csv.py` with all three submission CSVs
6. Submit the extended v22 (= v19+v21+v23 ensemble)

That's the path to top of LB.

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/rafaelproena/a3-adl"),
    Path("/kaggle/input/a3-adl"),
    Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if (p / "train.csv").exists()), None)
assert DATA_ROOT is not None, f"train.csv not found at any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================================
# MODELS LIST — append future versions here (v22 auto-skips missing ckpts)
# ============================================================================
# Each entry tells the loader:
#   backbone: 'efficientnet_b0' | 'resnet18' | 'resnet50' | 'densenet201'
#   input_size: training resolution (we resize eval inputs to this)
#   dropout: the value the model was trained with (only affects the rebuilt model;
#            inference uses eval mode so dropout is inactive anyway)
#   ckpt_candidates: list of paths to try, in order. First one that exists is used.
MODELS = [
    {
        "name": "v19_effnet_b0",
        "backbone": "efficientnet_b0",
        "input_size": 128,
        "dropout": 0.3,
        "ckpt_candidates": [
            "/kaggle/input/v19-ckpt/fulldata_best.pt",
            "/kaggle/input/v19-ckpt/runs/fulldata_best.pt",
            "/kaggle/input/datasets/rafaelproena/v19-ckpt/fulldata_best.pt",
            "/kaggle/working/runs/v19_fulldata_best.pt",
        ],
    },
    {
        "name": "v21_resnet50",
        "backbone": "resnet50",
        "input_size": 224,
        "dropout": 0.3,
        "ckpt_candidates": [
            "/kaggle/input/v21-ckpt/fulldata_best.pt",
            "/kaggle/input/v21-ckpt/runs/fulldata_best.pt",
            "/kaggle/input/datasets/rafaelproena/v21-ckpt/fulldata_best.pt",
            "/kaggle/working/runs/v21_fulldata_best.pt",
        ],
    },
    # === Future v23 model — uncomment and adjust when ckpt is ready ===
    # {
    #     "name": "v23_densenet201",
    #     "backbone": "densenet201",
    #     "input_size": 224,
    #     "dropout": 0.3,
    #     "ckpt_candidates": [
    #         "/kaggle/input/v23-ckpt/fulldata_best.pt",
    #     ],
    # },
]

# === Ensemble settings ===
USE_TEST_STAIN_NORM = True   # recompute pixel stats from test (same as v19/v21 training)
USE_ADABN           = True   # AdaBN pass on test before TTA
TTA_8WAY_D4         = True   # 8-way D4 TTA per model (matches v19/v21 training)
INFER_BATCH         = 96     # conservative — works for any backbone @ up to 256 input

# === Submission combo weights (set to None for equal weighting) ===
# If you know your component LBs, you can up-weight stronger models:
# e.g. {"v19_effnet_b0": 0.7455, "v21_resnet50": 0.76, "v23_densenet201": 0.78}
# Otherwise leave as None → equal weights.
MODEL_WEIGHTS_BY_NAME = None

NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

BASE_SEED = 1
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(BASE_SEED)

# v11 hardcoded stats (fallback if USE_TEST_STAIN_NORM=False)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

# Default dropout (only used as fallback when a ckpt lacks 'args.dropout')
DROPOUT = 0.3

# Resolve each MODEL's ckpt path (use first that exists).
def resolve_ckpt(candidates):
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    return None

print(f"\nv22 ensemble config:")
print(f"  USE_TEST_STAIN_NORM = {USE_TEST_STAIN_NORM}")
print(f"  USE_ADABN           = {USE_ADABN}")
print(f"  TTA_8WAY_D4         = {TTA_8WAY_D4}")
print(f"  INFER_BATCH         = {INFER_BATCH}")
print(f"\nResolving {len(MODELS)} model checkpoints:")
ACTIVE_MODELS = []
for m in MODELS:
    p = resolve_ckpt(m["ckpt_candidates"])
    if p is None:
        print(f"  [SKIP] {m['name']} — no ckpt found at any of {m['ckpt_candidates']}")
        continue
    m_copy = dict(m, ckpt_path=p)
    ACTIVE_MODELS.append(m_copy)
    print(f"  [OK]   {m['name']}  ({m['backbone']} @ {m['input_size']})  ->  {p}")

assert len(ACTIVE_MODELS) >= 1, "Need at least one model ckpt to ensemble!"
if len(ACTIVE_MODELS) == 1:
    print(f"\nWARNING: only 1 model active; output will be that single model's prediction (no ensembling).")
print(f"\n=> v22 will ensemble {len(ACTIVE_MODELS)} model(s)")

In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    """v19: also returns patient_id for MIL grouping in training."""
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
        # Patient ID is -1 for test rows (Name has no pat_X prefix in some splits but in this
        # dataset all names are pat_NN_image_MM.jpg so parse always succeeds).
        self.has_pid = "patient_id" in self.df.columns
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        pid = int(row["patient_id"]) if self.has_pid else -1
        return {"bf": bf, "fl": fl, "label": label, "patient_id": pid, "name": name}

class PatientBalancedSampler(Sampler):
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd

def _make_resnet50_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet50(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd

def _make_densenet201_branch(pretrained=True):
    """v23 candidate: DenseNet-201 — teacher's alternative backbone (Lu 2020)."""
    weights = "DEFAULT" if pretrained else None
    net = models.densenet201(weights=weights)
    # DenseNet stem is named 'features.conv0' (7x7 conv, 3-ch input).
    w = net.features.conv0.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.features.conv0 = new_conv
    fd = net.classifier.in_features  # 1920 for DenseNet-201
    net.classifier = nn.Identity()
    return net, fd

def _make_effnet_b0_branch(pretrained=True):
    import timm
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                            num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                        stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280

BRANCH_BUILDERS = {
    "resnet18":         _make_resnet18_branch,
    "resnet50":         _make_resnet50_branch,
    "ResNet-50":        _make_resnet50_branch,   # accept both stylings (v21 saved 'ResNet-50')
    "ResNet-18":        _make_resnet18_branch,
    "densenet201":      _make_densenet201_branch,
    "efficientnet_b0":  _make_effnet_b0_branch,
    "EfficientNet-B0":  _make_effnet_b0_branch,
}

class MultimodalClassifier(nn.Module):
    """Same head architecture as v19/v21 — dual branches + concat + 2-layer MLP head."""
    def __init__(self, backbone_name, pretrained=False, dropout=0.3):
        super().__init__()
        if backbone_name not in BRANCH_BUILDERS:
            raise ValueError(f"Unknown backbone '{backbone_name}'. "
                             f"Supported: {list(BRANCH_BUILDERS.keys())}")
        builder = BRANCH_BUILDERS[backbone_name]
        self.bf_branch, fd = builder(pretrained)
        self.fl_branch, _  = builder(pretrained)
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        self._feat_dim = fd
        self._backbone_name = backbone_name

    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

# Quick sanity check on every active model's backbone
print("Backbone builder sanity check:")
for m in ACTIVE_MODELS:
    with torch.no_grad():
        _m = MultimodalClassifier(m["backbone"], pretrained=False, dropout=m["dropout"]).cpu()
        _x = torch.zeros(2, 1, m["input_size"], m["input_size"])
        n_params = sum(p.numel() for p in _m.parameters()) / 1e6
        print(f"  {m['name']}: {m['backbone']} @ {m['input_size']}  "
              f"out_shape={tuple(_m(_x, _x).shape)}  params={n_params:.1f}M  feat_dim={_m._feat_dim}")
        del _m, _x

In [ ]:
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Test:  {len(df_test)} cells")

# For stain norm we sample test pixel stats. We DON'T need to cache the full train set
# because we're not training — but we do sample some train images for the print comparison.
print("\n--- Caching test JPEGs into RAM ---")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"Approx RAM used by test cache: {approx_mb:.0f} MB")

# Compute pixel statistics from test data for stain norm.
def _sample_pixel_stats(cache, names, n_sample=1500):
    rng = np.random.default_rng(42)
    sampled = rng.choice(names, size=min(n_sample, len(names)), replace=False)
    pixels = []
    for n in sampled:
        img = np.asarray(Image.open(io.BytesIO(cache[n])).convert("L"),
                         dtype=np.float32) / 255.0
        pixels.append(img.ravel())
    pixels = np.concatenate(pixels)
    return float(pixels.mean()), float(pixels.std())

if USE_TEST_STAIN_NORM:
    BF_MEAN_T, BF_STD_T = _sample_pixel_stats(bf_test_cache, df_test["Name"].tolist())
    FL_MEAN_T, FL_STD_T = _sample_pixel_stats(fl_test_cache, df_test["Name"].tolist())
    print(f"\nTest pixel statistics:")
    print(f"  BF test: mean={BF_MEAN_T:.4f} std={BF_STD_T:.4f}")
    print(f"  FL test: mean={FL_MEAN_T:.4f} std={FL_STD_T:.4f}")
    BF_MEAN, BF_STD = BF_MEAN_T, BF_STD_T
    FL_MEAN, FL_STD = FL_MEAN_T, FL_STD_T
    print(f"  -> using TEST stats (matches v19/v21 training)")
else:
    print(f"\nUsing v11 hardcoded normalization (USE_TEST_STAIN_NORM=False)")

# vFinal also needs the train cache for AdaBN if we want clean train statistics — but
# AdaBN runs on TEST batches (that's the whole point), so train cache is not needed.

In [ ]:
def make_eval_transform(modality, target_size):
    """Build an eval transform: ToTensor -> Resize(target_size) -> Normalize(BF or FL stats).
    For inference only — no aug. Each model has its own training resolution so we make
    a transform per (model, modality) pair.
    """
    mean = BF_MEAN if modality == "bf" else FL_MEAN
    std  = BF_STD  if modality == "bf" else FL_STD
    def fn(img):
        t = TF.to_tensor(img)
        if target_size != t.shape[-1]:
            t = TF.resize(t, [target_size, target_size], antialias=True)
        return TF.normalize(t, [mean], [std])
    return fn

def make_test_loader(target_size):
    """Build a DataLoader for the test set at the requested input resolution."""
    ds = CachedCellDataset(
        df_test, bf_test_cache, fl_test_cache,
        make_eval_transform("bf", target_size),
        make_eval_transform("fl", target_size),
        paired_tf=None,   # no aug at inference
    )
    return DataLoader(ds, batch_size=INFER_BATCH, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

# Pre-build one loader per unique input size (cache them — same loader works for all models
# that share the same input size).
UNIQUE_SIZES = sorted({m["input_size"] for m in ACTIVE_MODELS})
LOADERS = {sz: make_test_loader(sz) for sz in UNIQUE_SIZES}
print(f"Built test loaders for input sizes: {UNIQUE_SIZES}  (batch={INFER_BATCH})")
for sz, ldr in LOADERS.items():
    print(f"  size {sz}: {len(ldr)} batches x batch {INFER_BATCH}")

In [ ]:
# ============================================================================
# v22 core: load each model -> AdaBN -> 8-way D4 TTA -> save per-cell preds.
# Then combine across models in three ways.
# ============================================================================

@torch.no_grad()
def adabn_pass(model, loader):
    """Recalibrate BN running stats to test distribution."""
    model.train()
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            _ = model(bf, fl)
    model.eval()

def _d4(bf, fl):
    """8-way D4 augmentations: 4 rotations × 2 flips = 8."""
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def load_model_from_ckpt(ckpt_path, backbone_name, dropout):
    """Load a model from a checkpoint. The ckpt's 'args' dict is informational only —
    we use the explicitly-provided backbone_name and dropout because the ckpt's
    backbone string varies in casing ('resnet50' vs 'ResNet-50' etc).
    """
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    saved_dropout = args.get("dropout", dropout)
    model = MultimodalClassifier(backbone_name, pretrained=False,
                                 dropout=float(saved_dropout)).to(DEVICE)
    model.load_state_dict(state["model"])
    model.eval()
    return model, args

def predict_with_model(model_cfg):
    """Run AdaBN + 8-way D4 TTA on test set for one model. Returns numpy array of
    per-cell sigmoid predictions, in the order df_test rows are iterated.
    """
    print(f"\n--- Predicting with {model_cfg['name']} ---")
    print(f"  backbone={model_cfg['backbone']}  input={model_cfg['input_size']}  ckpt={model_cfg['ckpt_path']}")
    t0 = time.time()
    model, ckpt_args = load_model_from_ckpt(
        model_cfg["ckpt_path"], model_cfg["backbone"], model_cfg["dropout"])
    print(f"  ckpt args: {ckpt_args}  loaded in {time.time()-t0:.1f}s")
    loader = LOADERS[model_cfg["input_size"]]
    if USE_ADABN:
        t0 = time.time()
        adabn_pass(model, loader)
        print(f"  AdaBN pass done in {time.time()-t0:.1f}s")
    # 8-way D4 TTA
    t0 = time.time()
    preds = []
    n_aug = 8
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p_sum = None
            for bf_t, fl_t in _d4(bf, fl):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    p = torch.sigmoid(model(bf_t, fl_t)).float()
                p_sum = p if p_sum is None else p_sum + p
            preds.append((p_sum / n_aug).cpu().numpy())
    preds = np.concatenate(preds)
    print(f"  TTA done in {time.time()-t0:.1f}s   mean={preds.mean():.3f} "
          f"std={preds.std():.3f} min={preds.min():.4f} max={preds.max():.4f}")
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return preds  # shape (N_test,)

# Run inference for every active model
ALL_PREDS = {}   # name -> numpy array of sigmoid preds
for m in ACTIVE_MODELS:
    ALL_PREDS[m["name"]] = predict_with_model(m)

# Save individual model predictions as their own submission files too (for ablation)
for name, p in ALL_PREDS.items():
    sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": p})
    sub.to_csv(f"/kaggle/working/submission_{name}.csv", index=False)
    print(f"Saved /kaggle/working/submission_{name}.csv")
print(f"\nAll {len(ALL_PREDS)} model predictions computed.\n")

# ============================================================================
# Ensemble combinations (v22 = the average of these models)
# ============================================================================
from scipy.stats import rankdata  # for rank averaging

names = list(ALL_PREDS.keys())
P = np.stack([ALL_PREDS[n] for n in names], axis=0)  # shape (M_models, N_test)

# Resolve per-model weights
if MODEL_WEIGHTS_BY_NAME is None:
    weights = np.ones(len(names)) / len(names)
else:
    weights = np.array([MODEL_WEIGHTS_BY_NAME.get(n, 1.0) for n in names])
    weights = weights / weights.sum()
print(f"Model weights (normalized):")
for n, w in zip(names, weights):
    print(f"  {n}: {w:.4f}")

# 1. Sigmoid arithmetic mean
sigmoid_avg = (P * weights[:, None]).sum(axis=0)

# 2. Rank average
RANKS = np.zeros_like(P)
for i in range(P.shape[0]):
    # Convert each model's predictions to fractional ranks in [0, 1]
    RANKS[i] = rankdata(P[i], method="average") / len(P[i])
rank_avg = (RANKS * weights[:, None]).sum(axis=0)

# 3. Geometric mean of sigmoid outputs (clip to avoid log(0))
eps = 1e-7
P_clipped = np.clip(P, eps, 1 - eps)
log_geo = (np.log(P_clipped) * weights[:, None]).sum(axis=0)
geo_avg = np.exp(log_geo)

# Sanity prints
def stats(label, p):
    print(f"  {label}: mean={p.mean():.4f} std={p.std():.4f} "
          f"min={p.min():.4f} max={p.max():.4f}  "
          f">0.5: {(p > 0.5).mean():.3%}  <0.05: {(p < 0.05).mean():.3%}  >0.95: {(p > 0.95).mean():.3%}")

print(f"\nEnsemble combination stats:")
for n, p in zip(names, P):
    stats(f"(single) {n}", p)
print("  ---")
stats("sigmoid_avg", sigmoid_avg)
stats("rank_avg   ", rank_avg)
stats("geo_avg    ", geo_avg)

# Write three submission files
combos = {
    "sigmoid_avg": sigmoid_avg,
    "rank_avg":    rank_avg,
    "geomean":     geo_avg,
}
print(f"\nWriting submission files:")
for name, preds in combos.items():
    sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
    fn = f"/kaggle/working/submission_{name}.csv"
    sub.to_csv(fn, index=False)
    print(f"  {fn}  (mean={preds.mean():.3f})")

# Default submission.csv = sigmoid_avg (Kaggle will pick up this filename by default)
import shutil
shutil.copyfile("/kaggle/working/submission_sigmoid_avg.csv", "/kaggle/working/submission.csv")
print(f"\nDefault submission.csv = sigmoid_avg  (this is v22)")
!wc -l /kaggle/working/submission.csv

In [ ]:
# ============================================================================
# ALTERNATIVE: local CSV ensemble (no GPU / no Kaggle re-inference needed)
# ============================================================================
# Use this when you've already got submission.csv files from each version (v19, v21, ...)
# and just want to combine them. Faster than re-running inference but slightly less flexible
# (cannot re-AdaBN, cannot change TTA scheme).
#
# Example usage (run locally, paths to your downloaded submission CSVs):
#
#     local_csv_ensemble(
#         csv_paths=[
#             "results (1) v19/submission.csv",
#             "results (1) v21/submission.csv",   # or wherever v21 lands
#             # "results v22/submission.csv",
#         ],
#         weights=None,                    # equal weight, or e.g. [0.7455, 0.76]
#         out_dir="./ensemble_outputs",
#     )

import pandas as pd
import numpy as np
import os
from scipy.stats import rankdata

def local_csv_ensemble(csv_paths, weights=None, out_dir="./ensemble_outputs"):
    """Combine multiple submission.csv files using three strategies.
    Writes:
        out_dir/submission_sigmoid_avg.csv
        out_dir/submission_rank_avg.csv
        out_dir/submission_geomean.csv
    """
    os.makedirs(out_dir, exist_ok=True)
    dfs = [pd.read_csv(p) for p in csv_paths]
    # Sanity: same Name column order across all files
    n0 = dfs[0]["Name"].tolist()
    for d, p in zip(dfs[1:], csv_paths[1:]):
        if d["Name"].tolist() != n0:
            d = d.set_index("Name").reindex(n0).reset_index()
            print(f"[reindexed] {p} to match {csv_paths[0]} order")
            dfs[dfs.index(d) if d in dfs else len(dfs)-1] = d
    P = np.stack([d["Diagnosis"].values.astype(float) for d in dfs], axis=0)
    M = P.shape[0]
    if weights is None:
        w = np.ones(M) / M
    else:
        w = np.array(weights, dtype=float); w = w / w.sum()
    print(f"Combining {M} submissions with weights {w.tolist()}")

    sigmoid_avg = (P * w[:, None]).sum(axis=0)
    R = np.zeros_like(P)
    for i in range(M):
        R[i] = rankdata(P[i], method="average") / len(P[i])
    rank_avg = (R * w[:, None]).sum(axis=0)
    eps = 1e-7
    geo = np.exp((np.log(np.clip(P, eps, 1 - eps)) * w[:, None]).sum(axis=0))

    for name, preds in [("sigmoid_avg", sigmoid_avg),
                        ("rank_avg",    rank_avg),
                        ("geomean",     geo)]:
        out = pd.DataFrame({"Name": n0, "Diagnosis": preds})
        path = os.path.join(out_dir, f"submission_{name}.csv")
        out.to_csv(path, index=False)
        print(f"  wrote {path}  (mean={preds.mean():.3f} std={preds.std():.3f})")

print("local_csv_ensemble() is defined and ready. Call it from a Python shell or notebook.")